# BDAT05 · MIDTERM · Part 2A — Practical Notebook B
**Panaderia Buena Masa** — a Laguna bakeshop chain you have never seen before.
**60 minutes · open notes (your own notebooks and code) · NO AI of any kind.**

Five sections. Every section ends with a printed **EVIDENCE** result and a one-sentence **READING**.
Nothing changes silently — print your counts. A correct number with no reading earns nothing.

In [ ]:
# SETUP — run first. Your 60 minutes start here.
from datetime import datetime
from zoneinfo import ZoneInfo
STUDENT_NAME = ""      # Surname, First Name
NOTEBOOK = "B"
START_TIME = datetime.now(ZoneInfo("Asia/Manila"))
assert STUDENT_NAME.strip(), "Type your name before anything else."
print("Examinee:", STUDENT_NAME, "| Notebook", NOTEBOOK, "| started", START_TIME.strftime("%Y-%m-%d %H:%M"))

In [ ]:
# LOAD — paste the four FILE_IDs from the access sheet, then run.
import pandas as pd, numpy as np
FILE_ID_ORDERS   = ""
FILE_ID_BRANCHES = ""
FILE_ID_CALENDAR = ""
FILE_ID_PREDS    = ""
def _url(fid): return f"https://drive.google.com/uc?export=download&id={fid}"
orders   = pd.read_csv(_url(FILE_ID_ORDERS), dtype={"AmountPHP": str}, keep_default_na=False)
branches = pd.read_csv(_url(FILE_ID_BRANCHES))
calendar = pd.read_csv(_url(FILE_ID_CALENDAR), parse_dates=["Date"])
preds    = pd.read_csv(_url(FILE_ID_PREDS), parse_dates=["Date"])
print(orders.shape, branches.shape, calendar.shape, preds.shape)
orders.head(3)

---
## §1 · Prepare, then split — the calendar does the splitting  *(10 points)*

The file carries **five known defect families**: duplicated orders, dirty BranchID variants,
text-trapped amounts, blank dates, and orphan branch rows. Four are repaired for you below —
run and check the printed counts. **Your spotlight defect: duplicated orders.**
Some orders were exported twice. Count the rows involved, count the surplus, and keep exactly one copy of each OrderID.

Then split by the calendar: **training = everything before 2026-06-01 · holdout = 2026-06-01 onward.**

**EVIDENCE:** the count for your spotlight repair, the clean row count, and the train / holdout branch-day counts.
**READING:** one sentence — why does the calendar make this split, and not `sample()`?

In [ ]:
# GIVEN repairs — four of the five families. Run and check each count.
work = orders.copy()
print("raw rows:", len(work))

V = {"p01":"P01","P-02":"P02","P03.":"P03","p04":"P04","P 05":"P05","P06 ":"P06"}
work["BranchID"] = work["BranchID"].replace(V)
print("branch codes now:", sorted(work["BranchID"].unique()))

work["AmountPHP"] = work["AmountPHP"].str.replace(",", "").astype(float)
print("AmountPHP dtype:", work["AmountPHP"].dtype)

work = work[work["Date"] != ""]
work["Date"] = pd.to_datetime(work["Date"])
print("after date repair:", len(work))

work = work[work["BranchID"].isin(branches["BranchID"])]
print("after orphan removal:", len(work))

In [ ]:
# YOUR spotlight repair — duplicated orders.
# Investigate, count, repair. Print the count BEFORE you repair it.
# ...your code...


In [ ]:
# EVIDENCE §1 — run after your repair. All five families must be repaired by this point.
clean = work.copy()
print("EVIDENCE clean rows:", len(clean))
bd = clean.groupby(["BranchID","Date"], as_index=False)["AmountPHP"].sum().rename(columns={"AmountPHP":"Revenue"})
SPLIT = pd.Timestamp("2026-06-01")
train, hold = bd[bd["Date"] < SPLIT], bd[bd["Date"] >= SPLIT]
print("EVIDENCE branch-days:", len(bd), "| train:", len(train), "| holdout:", len(hold))

**READING §1:**  
*(one sentence — why the calendar, not `sample()`)*

> 

---
## §2 · Features and the leakage test  *(15 points)*

The test is one question: **was this knowable at the moment the decision would be taken?**
The decision: *predict a branch-day's revenue, the evening before.*

**(a) Build your assigned feature — `PrevDaySameBranch`.** For each branch-day, build PrevDayRevenue = that branch's revenue one day earlier. EVIDENCE: correlation between PrevDayRevenue and Revenue on the training window.

**(b) The candidate table.** Classify each candidate **KEEP** or **LEAK**, with a calendar reason.

| # | Candidate feature | What the column is |
|---|---|---|
| 1 | DayOfWeek of the day being predicted | calendar fact, fixed in advance |
| 2 | PromoPlanned flag for the day being predicted | planned by head office weeks ahead |
| 3 | The same branch's revenue 7 days earlier | already finished and recorded |
| 4 | ItemsSold on the day being predicted | counted only as the day happens |
| 5 | Month-to-date revenue including the day being predicted | contains the target day inside the sum |
| 6 | Refunds processed on the day being predicted | processed during and after the day |


In [ ]:
# (a) Build PrevDaySameBranch and print its EVIDENCE.
# ...your code...


**(b) Verdicts — fill every row:**

| # | KEEP or LEAK | Calendar reason (one clause) |
|---|---|---|
| 1 |  |  |
| 2 |  |  |
| 3 |  |  |
| 4 |  |  |
| 5 |  |  |
| 6 |  |  |

**READING §2:** *(one sentence — the feature test, in your own words)*

> 

---
## §3 · Baseline first — you are not allowed to beat nothing  *(10 points)*

Before looking at any model: compute the **naive-7 baseline** — predict each holdout branch-day
with the same branch's revenue **7 days earlier** — and score it with **MAE** on both of your assigned branches.
Your assigned slice: **P02 and P04**.

**EVIDENCE:** the baseline MAE, printed per assigned slice.
**READING:** one sentence — why is this baseline computed *before* the model is examined?

In [ ]:
# §3 — naive-7 baseline on the holdout.
ASSIGNED = ['P02', 'P04']
bdi = bd.set_index(["BranchID","Date"])["Revenue"]
ho = hold.copy()
ho["Naive7"] = [bdi.get((r.BranchID, r.Date - pd.Timedelta(days=7)), np.nan) for r in ho.itertuples()]
ho = ho.dropna(subset=["Naive7"])
def mae(a, f): return float(np.mean(np.abs(a - f)))
for tag in ASSIGNED:
    g = ho if tag == "ALL" else ho[ho["BranchID"] == tag]
    print(f"EVIDENCE baseline MAE [{tag}]:", round(mae(g["Revenue"], g["Naive7"]), 2))

**READING §3:**

> 

---
## §4 · The model must earn it  *(15 points)*

Head office bought **“Model M,”** a forecasting service. Its predictions for every holdout
branch-day are in `preds`. Score Model M with MAE on your assigned slice —
**P02 and P04** — beside your §3 baseline, and compute the ratio
**MAE(Model M) ÷ MAE(naive-7)**. Below 1.0 = the model earns its keep on that slice.

**EVIDENCE:** both MAEs and the ratio, per assigned slice, with **WINS / LOSES** printed.
**READING:** one sentence per slice — what the ratio says, in pesos-and-decisions language.

In [ ]:
# §4 — Model M vs your baseline.
ev = ho.merge(preds, on=["BranchID","Date"], how="left").dropna(subset=["ModelM_PredictedRevenue"])
for tag in ASSIGNED:
    g = ev if tag == "ALL" else ev[ev["BranchID"] == tag]
    m_b = mae(g["Revenue"], g["Naive7"]); m_m = mae(g["Revenue"], g["ModelM_PredictedRevenue"])
    r = m_m / m_b
    print(f"EVIDENCE [{tag}] baseline {m_b:,.2f} | Model M {m_m:,.2f} | ratio {r:.3f} |", "MODEL WINS" if r < 1 else "MODEL LOSES")

**READING §4:**

> 

---
## §5 · Judgment — the memo line  *(5 points)*

Head office asks one question: **“Do we roll Model M out to every branch?”**

Write your verdict in **three sentences maximum**, using only numbers your own notebook printed:
what you recommend, which evidence carries it, and what you would check next.
A verdict your own evidence does not support earns nothing.

**MEMO:**

> 

In [ ]:
# FINISH — run last.
from datetime import datetime
from zoneinfo import ZoneInfo
END_TIME = datetime.now(ZoneInfo("Asia/Manila"))
def _surname(name):
    if "," in name: return name.split(",")[0].strip().upper()
    parts = [p for p in name.split() if p.strip(".")]
    return (parts[-1] if parts else "SURNAME").upper()
mins = (END_TIME - START_TIME).total_seconds() / 60
print(f"Examinee: {STUDENT_NAME} | Notebook {NOTEBOOK} | elapsed {mins:.0f} min")
print(f"Save as: BDAT05_Midterm_Part2A_{NOTEBOOK}_{_surname(STUDENT_NAME)}.ipynb and upload to your midterm folder.")